# 1. Load Raw Data and Create Cleaning Copies

## Purpose

This section starts the data-cleaning stage by loading the original Rossmann datasets again from the raw data directory.

The cleaning notebook is designed to run independently from `01_check.ipynb`. Therefore, it does not reuse DataFrames remaining in memory from the inspection notebook.

Two separate working copies are created:

- `train_clean`
- `store_clean`

All cleaning operations in this notebook will be applied to these working copies rather than directly to the initially loaded raw DataFrames.

## Input

The code reads the following original files:

- `data/raw/train.csv`
- `data/raw/store.csv`

These files are treated as immutable source data and are not modified.

## Output

No output file is created in this step.

Four Pandas DataFrames are created in memory:

- `train`: the raw daily sales dataset;
- `store`: the raw store-information dataset;
- `train_clean`: the working copy used for cleaning;
- `store_clean`: the working copy used for cleaning.

The cleaned datasets will only be written to `data/processed/` after all cleaning steps and validation checks have been completed.

## Method

`pathlib.Path` is used to construct project-relative file paths, and `pandas.read_csv()` is used to load the CSV files.

The raw DataFrames are then duplicated using `DataFrame.copy()`.

A separate working copy is used because cleaning operations modify data. Keeping the initially loaded raw DataFrames unchanged provides an in-memory reference that can later be used to compare the data before and after cleaning.

The raw CSV files are not overwritten because preserving the original source data is necessary for reproducibility.

## Principle

A reproducible cleaning workflow should separate:

**Raw data → Working copy → Cleaned data**

The raw data represents the original evidence and should remain unchanged.

Cleaning transformations are applied to a separate working dataset. This allows each transformation to be traced and validated without destroying the original representation.

The processed dataset should only be saved after all intended transformations have been completed and checked.

## Cleaning Strategy

Based on the findings from `01_check.ipynb`, this notebook will:

1. normalize the inconsistent representation of `StateHoliday`;
2. convert `Date` to a proper datetime type;
3. preserve the structural meaning of missing `Promo2` detail fields;
4. handle competition-related missing information without inventing artificial values;
5. retain rare zero-sales review cases unless stronger evidence indicates an error;
6. retain statistical IQR outliers unless they are independently shown to be invalid.

Unnecessary row deletion and automatic statistical imputation will be avoided.

## Evaluation Criteria

This initialization step is successful if:

- both raw files can be loaded;
- both working copies have exactly the same initial dimensions as their corresponding raw datasets;
- the original DataFrames and working copies are independent objects;
- no output file or raw source file has been modified.

Subsequent sections will apply and validate each cleaning transformation separately.

In [1]:
# Load the raw datasets and create independent working copies for cleaning.

from pathlib import Path
import pandas as pd


# ------------------------------------------------------------
# 1. Define project-relative paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.csv"
STORE_PATH = PROJECT_ROOT / "data" / "raw" / "store.csv"

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"


# ------------------------------------------------------------
# 2. Validate required input files
# ------------------------------------------------------------

if not TRAIN_PATH.exists():
    raise FileNotFoundError(
        f"train.csv was not found: {TRAIN_PATH}"
    )

if not STORE_PATH.exists():
    raise FileNotFoundError(
        f"store.csv was not found: {STORE_PATH}"
    )


# ------------------------------------------------------------
# 3. Load the original datasets
# ------------------------------------------------------------

train = pd.read_csv(TRAIN_PATH)
store = pd.read_csv(STORE_PATH)


# ------------------------------------------------------------
# 4. Create independent cleaning copies
# ------------------------------------------------------------

train_clean = train.copy()
store_clean = store.copy()


# ------------------------------------------------------------
# 5. Basic initialization validation
# ------------------------------------------------------------

initialization_summary = pd.DataFrame({
    "Dataset": ["train", "store"],
    "Raw rows": [
        train.shape[0],
        store.shape[0]
    ],
    "Raw columns": [
        train.shape[1],
        store.shape[1]
    ],
    "Cleaning rows": [
        train_clean.shape[0],
        store_clean.shape[0]
    ],
    "Cleaning columns": [
        train_clean.shape[1],
        store_clean.shape[1]
    ],
    "Same object": [
        train is train_clean,
        store is store_clean
    ]
})

display(initialization_summary)

C:\Users\31729\AppData\Local\Temp\ipykernel_22708\3769313536.py:38: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv(TRAIN_PATH)


,Dataset,Raw rows,Raw columns,Cleaning rows,Cleaning columns,Same object
0,train,1017209,9,1017209,9,False
1,store,1115,10,1115,10,False


# 2. Normalize the `StateHoliday` Representation

## Observation

The inspection stage identified a confirmed representation inconsistency in `StateHoliday`.

The no-holiday category was stored in two different Python types:

- string `"0"` in 855,087 records;
- integer `0` in 131,072 records.

The holiday categories `a`, `b`, and `c` were stored as strings.

All observed categories were semantically valid, so the problem concerns inconsistent representation rather than invalid category values.

## Purpose

This section normalizes `StateHoliday` so that every category is represented using the same data type.

The transformation is applied only to `train_clean`. The original `train` DataFrame remains unchanged and can therefore be used as a reference for before-and-after validation.

## Input

The transformation uses the working DataFrame:

- `train_clean`, originally loaded from `data/raw/train.csv`

The source file itself is not modified.

## Output

No file is written in this step.

The following in-memory object is modified:

- `train_clean["StateHoliday"]`

The column is converted to Pandas' dedicated `string` data type.

The code also produces:

1. a profile of the original representations;
2. a profile of the normalized categories;
3. a validation summary comparing the column before and after cleaning.

## Method

`astype("string")` is used to convert all non-missing values in `StateHoliday` to a consistent string representation.

As a result:

- integer `0` becomes string `"0"`;
- existing string `"0"` remains `"0"`;
- `a`, `b`, and `c` remain unchanged.

The Pandas `string` data type is preferred to leaving the column as `object` because `object` can contain mixed Python object types, while `string` explicitly represents textual data.

The column is not converted to numeric values because `a`, `b`, and `c` are categorical labels rather than numbers.

It is also not converted to numerical category codes at this stage because numerical encoding is a modelling decision and will be handled later during feature preparation.

## Principle

Data cleaning should preserve the semantic meaning of a variable while removing unnecessary representational inconsistency.

Before cleaning, the following two values represent the same business category but are different Python objects:

- `0`
- `"0"`

After conversion, both are represented as:

- `"0"`

Therefore, the number of semantic categories remains unchanged even though the number of distinct Python representations decreases.

This process is a form of **categorical normalization**.

The objective is not to change what the data means, but to ensure that the same category is represented consistently throughout the dataset.

## Evaluation Criteria

The transformation is considered successful if:

1. the number of rows remains unchanged;
2. the number of missing values remains unchanged;
3. the resulting data type is `string`;
4. only the categories `0`, `a`, `b`, and `c` remain;
5. no invalid category is introduced;
6. the combined number of integer `0` and string `"0"` values before cleaning equals the number of string `"0"` values after cleaning.

If all conditions are satisfied, the original mixed-type representation issue is considered resolved.

In [2]:
# Normalize StateHoliday to one consistent categorical representation.

# ------------------------------------------------------------
# 1. Inspect representations before cleaning
# ------------------------------------------------------------

stateholiday_before = train_clean["StateHoliday"]

before_profile = []

for value in stateholiday_before.drop_duplicates():

    if pd.isna(value):
        count = stateholiday_before.isna().sum()
    else:
        count = (stateholiday_before == value).sum()

    before_profile.append({
        "Value (repr)": repr(value),
        "Python type": type(value).__name__,
        "Count": int(count)
    })

before_profile = (
    pd.DataFrame(before_profile)
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)

print("Before normalization:")
display(before_profile)


# Record values required for validation.
rows_before = len(train_clean)

missing_before = int(
    stateholiday_before.isna().sum()
)

zero_count_before = int(
    (
        (stateholiday_before == 0)
        | (stateholiday_before == "0")
    ).sum()
)


# ------------------------------------------------------------
# 2. Normalize the column
# ------------------------------------------------------------

train_clean["StateHoliday"] = (
    train_clean["StateHoliday"]
    .astype("string")
)


# ------------------------------------------------------------
# 3. Inspect categories after cleaning
# ------------------------------------------------------------

stateholiday_after = train_clean["StateHoliday"]

after_profile = (
    stateholiday_after
    .value_counts(dropna=False)
    .rename_axis("StateHoliday")
    .reset_index(name="Count")
)

print("\nAfter normalization:")
display(after_profile)


# ------------------------------------------------------------
# 4. Validate the transformation
# ------------------------------------------------------------

allowed_categories = {
    "0",
    "a",
    "b",
    "c"
}

invalid_after = int(
    (
        stateholiday_after.notna()
        & ~stateholiday_after.isin(allowed_categories)
    ).sum()
)

zero_count_after = int(
    (stateholiday_after == "0").sum()
)

validation_summary = pd.DataFrame({
    "Check": [
        "Rows preserved",
        "Missing values preserved",
        "Resulting dtype",
        "Invalid categories",
        "No-holiday count preserved"
    ],
    "Result": [
        len(train_clean) == rows_before,
        int(stateholiday_after.isna().sum()) == missing_before,
        str(stateholiday_after.dtype),
        invalid_after,
        zero_count_after == zero_count_before
    ]
})

print("\nNormalization validation:")
display(validation_summary)

Before normalization:


,Value (repr),Python type,Count
0,'0',str,855087
1,0,int,131072
2,'a',str,20260
3,'b',str,6690
4,'c',str,4100



After normalization:


,StateHoliday,Count
0,0,986159
1,a,20260
2,b,6690
3,c,4100



Normalization validation:


,Check,Result
0,Rows preserved,True
1,Missing values preserved,True
2,Resulting dtype,string
3,Invalid categories,0
4,No-holiday count preserved,True


# 3. Convert `Date` to Datetime

## Observation

The inspection stage confirmed that all 1,017,209 values in `Date` can be parsed using the expected `YYYY-MM-DD` format.

The date range is from `2013-01-01` to `2015-07-31`, with 942 consecutive calendar dates and no inconsistencies between `Date` and `DayOfWeek`.

Therefore, the original date values are valid, but the column is currently stored as a string rather than a datetime variable.

## Purpose

This section converts `train_clean["Date"]` from its original string representation to Pandas' datetime type.

The transformation is required because later stages of the project will need calendar operations such as:

- extracting year and month;
- sorting observations chronologically;
- aggregating daily records into monthly records;
- creating time-based modelling features.

The original `train["Date"]` column remains unchanged.

## Input

The transformation uses:

- `train_clean["Date"]`

which originates from:

- `data/raw/train.csv`

No additional file is read.

## Output

No file is written in this step.

The in-memory column:

- `train_clean["Date"]`

is converted from a string-based type to `datetime64[ns]`.

A validation table is produced to confirm that:

- the row count is unchanged;
- the number of missing values is unchanged;
- the resulting data type is datetime;
- the minimum and maximum dates are preserved;
- `DayOfWeek` remains fully consistent with the converted dates.

## Method

`pd.to_datetime()` is used with the explicit format:

`%Y-%m-%d`

which corresponds to:

`YYYY-MM-DD`

Unlike the inspection stage, `errors="raise"` is used during cleaning.

The inspection notebook previously used `errors="coerce"` because its purpose was to detect invalid values without stopping execution. Since all dates have now been verified as valid, the cleaning step uses `errors="raise"` so that any unexpected conversion failure immediately stops the workflow.

This fail-fast approach prevents unnoticed invalid dates from being introduced into the cleaned dataset.

## Principle

A date stored as text and a true datetime value may display similarly but behave differently.

A string such as:

`"2015-07-31"`

is treated primarily as text.

After conversion, the same value is represented as a calendar-aware datetime value, allowing operations such as:

- extracting `.dt.year`;
- extracting `.dt.month`;
- determining weekdays;
- calculating time intervals;
- performing chronological aggregation.

The conversion changes the data representation while preserving the underlying calendar information.

## Evaluation Criteria

The transformation is considered successful if:

1. all rows are preserved;
2. no new missing values are introduced;
3. the resulting column has a datetime data type;
4. the earliest date remains `2013-01-01`;
5. the latest date remains `2015-07-31`;
6. all converted dates remain consistent with the existing `DayOfWeek` field.

If all checks pass, the `Date` column is ready for later monthly aggregation and time-based feature engineering.

In [4]:
# Convert Date from string representation to Pandas datetime.

# ------------------------------------------------------------
# 1. Record the original state for validation
# ------------------------------------------------------------

date_before = train_clean["Date"].copy()

rows_before_date_conversion = len(train_clean)

missing_before_date_conversion = int(
    date_before.isna().sum()
)

min_date_before = date_before.min()
max_date_before = date_before.max()


# ------------------------------------------------------------
# 2. Convert Date to datetime
# ------------------------------------------------------------

train_clean["Date"] = pd.to_datetime(
    train_clean["Date"],
    format="%Y-%m-%d",
    errors="raise"
)


# ------------------------------------------------------------
# 3. Validate the converted dates
# ------------------------------------------------------------

date_after = train_clean["Date"]

expected_day_of_week = (
    date_after.dt.dayofweek + 1
)

weekday_mismatches_after = int(
    (
        train_clean["DayOfWeek"]
        != expected_day_of_week
    ).sum()
)


date_conversion_validation = pd.DataFrame({
    "Check": [
        "Rows preserved",
        "Missing values preserved",
        "Original dtype",
        "Resulting dtype",
        "Earliest date preserved",
        "Latest date preserved",
        "DayOfWeek mismatches"
    ],
    "Result": [
        len(train_clean) == rows_before_date_conversion,
        int(date_after.isna().sum()) == missing_before_date_conversion,
        str(date_before.dtype),
        str(date_after.dtype),
        date_after.min().strftime("%Y-%m-%d") == min_date_before,
        date_after.max().strftime("%Y-%m-%d") == max_date_before,
        weekday_mismatches_after
    ]
})

display(date_conversion_validation)

,Check,Result
0,Rows preserved,True
1,Missing values preserved,True
2,Original dtype,str
3,Resulting dtype,datetime64[us]
4,Earliest date preserved,True
5,Latest date preserved,True
6,DayOfWeek mismatches,0


# 4. Preserve Structural Missingness in `Promo2` Variables

## Observation

The inspection stage showed a fully consistent missing-value pattern in the `Promo2` variables.

Among the 1,115 stores:

- 544 stores have `Promo2 = 0`;
- all 544 of these stores have missing values in `Promo2SinceWeek`, `Promo2SinceYear`, and `PromoInterval`;
- 571 stores have `Promo2 = 1`;
- all 571 of these stores have complete values for the three `Promo2` detail fields.

This demonstrates that the missing values are structurally determined by participation in `Promo2` rather than representing accidental data loss.

## Purpose

This section preserves the "not applicable" meaning of the missing `Promo2` detail fields while normalizing their data types.

No statistical imputation is applied.

The following type normalization is performed:

- `Promo2SinceWeek` → nullable integer;
- `Promo2SinceYear` → nullable integer;
- `PromoInterval` → Pandas string.

Missing values remain missing.

## Input

The transformation uses:

- `store_clean`

which originates from:

- `data/raw/store.csv`

The relevant fields are:

- `Promo2`
- `Promo2SinceWeek`
- `Promo2SinceYear`
- `PromoInterval`

No additional file is read.

## Output

No file is written in this step.

The following columns in `store_clean` are normalized:

- `Promo2SinceWeek`
- `Promo2SinceYear`
- `PromoInterval`

Their missing-value structure is preserved.

A validation table is produced to confirm that:

- no missing values were added or removed;
- all `Promo2 = 0` stores still have missing detail fields;
- all `Promo2 = 1` stores still have complete detail fields;
- the resulting data types are appropriate for the meaning of each variable.

## Method

The numeric `Promo2` detail fields are converted to Pandas' nullable integer type (`Int64`).

This type is used instead of ordinary `int64` because ordinary integer columns cannot represent missing values directly, whereas nullable `Int64` supports both whole numbers and `pd.NA`.

`PromoInterval` is converted to Pandas' `string` type while preserving its missing values.

No `fillna()` operation is applied.

In particular, missing values are not replaced with:

- 0;
- the mean;
- the median;
- the mode.

These approaches would incorrectly assign meaningful values to stores for which the corresponding `Promo2` variables are not applicable.

## Principle

Missingness can contain information.

For a store with:

`Promo2 = 0`

the questions "When did Promo2 start?" and "In which months is Promo2 active?" are not applicable.

Therefore:

`Promo2SinceWeek = missing`

does not mean that the starting week is unknown. It means that no starting week exists because the store does not participate in the promotion.

Replacing this missing value with a numerical value would change the meaning of the data.

The appropriate cleaning strategy is therefore to preserve the missingness while using data types that represent the variables correctly.

This is an example of **semantic preservation during data cleaning**.

## Evaluation Criteria

The transformation is considered successful if:

1. the number of stores remains 1,115;
2. exactly 544 stores with `Promo2 = 0` retain missing values in all three detail fields;
3. all 571 stores with `Promo2 = 1` retain complete detail fields;
4. no partial missing pattern is introduced;
5. `Promo2SinceWeek` and `Promo2SinceYear` use nullable integer types;
6. `PromoInterval` uses the Pandas string type;
7. no artificial values are introduced.

If these conditions are satisfied, the structural missingness is considered correctly preserved.

In [5]:
# Preserve structural missingness in Promo2 detail fields
# while normalizing their data types.

promo2_detail_cols = [
    "Promo2SinceWeek",
    "Promo2SinceYear",
    "PromoInterval"
]


# ------------------------------------------------------------
# 1. Record the state before normalization
# ------------------------------------------------------------

rows_before_promo2 = len(store_clean)

missing_before_promo2 = (
    store_clean[promo2_detail_cols]
    .isna()
    .sum()
)


# ------------------------------------------------------------
# 2. Normalize data types without filling missing values
# ------------------------------------------------------------

store_clean["Promo2SinceWeek"] = (
    store_clean["Promo2SinceWeek"]
    .astype("Int64")
)

store_clean["Promo2SinceYear"] = (
    store_clean["Promo2SinceYear"]
    .astype("Int64")
)

store_clean["PromoInterval"] = (
    store_clean["PromoInterval"]
    .astype("string")
)


# ------------------------------------------------------------
# 3. Recheck the structural missing-value pattern
# ------------------------------------------------------------

promo2_zero = store_clean["Promo2"] == 0
promo2_one = store_clean["Promo2"] == 1

all_details_missing = (
    store_clean[promo2_detail_cols]
    .isna()
    .all(axis=1)
)

all_details_present = (
    store_clean[promo2_detail_cols]
    .notna()
    .all(axis=1)
)

partial_missing = (
    store_clean[promo2_detail_cols]
    .isna()
    .any(axis=1)
    & ~all_details_missing
)


# ------------------------------------------------------------
# 4. Validate the transformation
# ------------------------------------------------------------

missing_after_promo2 = (
    store_clean[promo2_detail_cols]
    .isna()
    .sum()
)

promo2_validation = pd.DataFrame({
    "Check": [
        "Rows preserved",
        "Missing counts preserved",
        "Promo2=0 stores",
        "Promo2=0 with all details missing",
        "Promo2=1 stores",
        "Promo2=1 with all details present",
        "Partial missing patterns",
        "Promo2SinceWeek dtype",
        "Promo2SinceYear dtype",
        "PromoInterval dtype"
    ],
    "Result": [
        len(store_clean) == rows_before_promo2,
        missing_before_promo2.equals(missing_after_promo2),
        int(promo2_zero.sum()),
        int((promo2_zero & all_details_missing).sum()),
        int(promo2_one.sum()),
        int((promo2_one & all_details_present).sum()),
        int(partial_missing.sum()),
        str(store_clean["Promo2SinceWeek"].dtype),
        str(store_clean["Promo2SinceYear"].dtype),
        str(store_clean["PromoInterval"].dtype)
    ]
})

display(promo2_validation)

,Check,Result
0,Rows preserved,True
1,Missing counts preserved,True
2,Promo2=0 stores,544
3,Promo2=0 with all details missing,544
4,Promo2=1 stores,571
5,Promo2=1 with all details present,571
6,Partial missing patterns,0
7,Promo2SinceWeek dtype,Int64
8,Promo2SinceYear dtype,Int64
9,PromoInterval dtype,string


# 5. Preserve and Normalize Missing Competition Information

## Observation

The inspection stage identified missing values in three competition-related variables:

- `CompetitionDistance`: 3 missing values;
- `CompetitionOpenSinceMonth`: 354 missing values;
- `CompetitionOpenSinceYear`: 354 missing values.

The opening month and year are always missing together.

Among the 354 stores with missing competition opening dates, 351 still have a recorded `CompetitionDistance`. This indicates that a nearby competitor is known to exist, but its opening date is unavailable.

The remaining 3 stores are also missing `CompetitionDistance`.

Unlike the missing `Promo2` detail fields, these values are not structurally "not applicable". They represent unavailable or unknown information.

## Purpose

This section preserves the unknown meaning of the competition-related missing values while normalizing the data types of the affected columns.

The following type normalization is performed:

- `CompetitionDistance` → nullable floating-point type;
- `CompetitionOpenSinceMonth` → nullable integer type;
- `CompetitionOpenSinceYear` → nullable integer type.

No missing value is imputed in this stage.

## Input

The transformation uses:

- `store_clean`

which originates from:

- `data/raw/store.csv`

The relevant variables are:

- `CompetitionDistance`
- `CompetitionOpenSinceMonth`
- `CompetitionOpenSinceYear`

No additional file is read.

## Output

No file is written in this step.

The competition-related columns in `store_clean` are normalized while preserving their existing missing values.

A validation table is produced to confirm:

- the number of rows is unchanged;
- the number of missing values in each field is unchanged;
- month and year remain missing together;
- no partial month/year missing pattern is introduced;
- the resulting data types match the logical meaning of the variables.

## Method

`CompetitionOpenSinceMonth` and `CompetitionOpenSinceYear` are converted to Pandas' nullable integer type (`Int64`).

Although these variables were originally loaded as `float64`, all observed non-missing values were verified during data inspection to be whole numbers. Nullable integers therefore represent their meaning more accurately while still allowing `pd.NA`.

`CompetitionDistance` is converted to Pandas' nullable floating-point type (`Float64`). Distance is a continuous numerical quantity and may legitimately contain non-integer values in other datasets, so it is retained as a floating-point variable.

No `fillna()` operation is applied.

Mean, median, zero, or other fixed-value imputation is intentionally avoided because the appropriate modelling treatment of genuinely unknown competition information should be decided later during feature engineering and preprocessing.

## Principle

A cleaned dataset does not necessarily contain no missing values.

Cleaning should preserve uncertainty when the true value is genuinely unknown.

For example, if a store has:

`CompetitionDistance = 500`

but:

`CompetitionOpenSinceMonth = missing`

`CompetitionOpenSinceYear = missing`

the available evidence supports the existence and distance of a competitor but does not reveal when that competitor opened.

Assigning an artificial date would create information that is not present in the original data.

This differs from structural missingness in the `Promo2` variables. In this case, the value is applicable but unknown.

Therefore, the correct cleaning action is to preserve the missing value and normalize its representation rather than invent a replacement.

## Evaluation Criteria

The transformation is considered successful if:

1. all 1,115 store records are preserved;
2. `CompetitionDistance` still contains exactly 3 missing values;
3. `CompetitionOpenSinceMonth` still contains exactly 354 missing values;
4. `CompetitionOpenSinceYear` still contains exactly 354 missing values;
5. opening month and year remain missing together;
6. no month-only or year-only missing pattern is introduced;
7. `CompetitionDistance` uses nullable `Float64`;
8. the opening month and year use nullable `Int64`;
9. no artificial competition values are introduced.

The final modelling treatment of these missing values will be decided later during feature engineering and preprocessing.

In [6]:
# Preserve genuinely unknown competition information
# while normalizing the affected data types.

competition_cols = [
    "CompetitionDistance",
    "CompetitionOpenSinceMonth",
    "CompetitionOpenSinceYear"
]


# ------------------------------------------------------------
# 1. Record the original cleaning-state information
# ------------------------------------------------------------

rows_before_competition = len(store_clean)

missing_before_competition = (
    store_clean[competition_cols]
    .isna()
    .sum()
)


# ------------------------------------------------------------
# 2. Normalize data types without imputing missing values
# ------------------------------------------------------------

store_clean["CompetitionDistance"] = (
    store_clean["CompetitionDistance"]
    .astype("Float64")
)

store_clean["CompetitionOpenSinceMonth"] = (
    store_clean["CompetitionOpenSinceMonth"]
    .astype("Int64")
)

store_clean["CompetitionOpenSinceYear"] = (
    store_clean["CompetitionOpenSinceYear"]
    .astype("Int64")
)


# ------------------------------------------------------------
# 3. Recheck the missing-value structure
# ------------------------------------------------------------

distance_missing = (
    store_clean["CompetitionDistance"]
    .isna()
)

month_missing = (
    store_clean["CompetitionOpenSinceMonth"]
    .isna()
)

year_missing = (
    store_clean["CompetitionOpenSinceYear"]
    .isna()
)

both_open_date_missing = (
    month_missing & year_missing
)

only_month_missing = (
    month_missing & ~year_missing
)

only_year_missing = (
    ~month_missing & year_missing
)

distance_available_date_missing = (
    ~distance_missing
    & month_missing
    & year_missing
)

all_competition_missing = (
    distance_missing
    & month_missing
    & year_missing
)


# ------------------------------------------------------------
# 4. Validate the transformation
# ------------------------------------------------------------

missing_after_competition = (
    store_clean[competition_cols]
    .isna()
    .sum()
)

competition_validation = pd.DataFrame({
    "Check": [
        "Rows preserved",
        "Missing counts preserved",
        "CompetitionDistance missing",
        "Opening month missing",
        "Opening year missing",
        "Month and year both missing",
        "Only month missing",
        "Only year missing",
        "Distance available but date missing",
        "All competition fields missing",
        "CompetitionDistance dtype",
        "CompetitionOpenSinceMonth dtype",
        "CompetitionOpenSinceYear dtype"
    ],
    "Result": [
        len(store_clean) == rows_before_competition,
        missing_before_competition.equals(
            missing_after_competition
        ),
        int(distance_missing.sum()),
        int(month_missing.sum()),
        int(year_missing.sum()),
        int(both_open_date_missing.sum()),
        int(only_month_missing.sum()),
        int(only_year_missing.sum()),
        int(distance_available_date_missing.sum()),
        int(all_competition_missing.sum()),
        str(store_clean["CompetitionDistance"].dtype),
        str(store_clean["CompetitionOpenSinceMonth"].dtype),
        str(store_clean["CompetitionOpenSinceYear"].dtype)
    ]
})

display(competition_validation)

,Check,Result
0,Rows preserved,True
1,Missing counts preserved,True
2,CompetitionDistance missing,3
3,Opening month missing,354
4,Opening year missing,354
5,Month and year both missing,354
6,Only month missing,0
7,Only year missing,0
8,Distance available but date missing,351
9,All competition fields missing,3


# 6. Validate the Cleaned Datasets

## Purpose

Before the cleaned datasets are written to disk, this section performs a final validation of the in-memory cleaning results.

The purpose is not to repeat the full data-quality investigation from `01_check.ipynb`.

Instead, this section verifies that the intended cleaning transformations were completed successfully and that no unintended structural changes were introduced.

The validation focuses on the main invariants that should still hold after cleaning.

## Input

This section uses the four DataFrames currently held in memory:

- `train`
- `store`
- `train_clean`
- `store_clean`

The raw DataFrames are used as reference objects, while the cleaned DataFrames contain the transformations performed in this notebook.

No additional file is read.

## Output

No file is written in this step.

The code produces a final cleaning-validation table covering:

- row and column preservation;
- duplicate and business-key integrity;
- `StateHoliday` normalization;
- `Date` conversion;
- missing-value structure;
- store-ID consistency;
- preservation of fields that were not intended to change.

A second table displays the final data types of both cleaned datasets.

## Method

The validation combines several deterministic checks.

Structural integrity is tested using:

- DataFrame dimensions;
- `duplicated()`;
- business-key duplication rules.

Type normalization is checked using:

- the resulting Pandas data types;
- allowed category sets;
- datetime-type detection.

Missing-value patterns are compared with the known results established during inspection.

Columns that were not intentionally transformed are compared directly with their original versions using `DataFrame.equals()`.

This method is preferred to visually inspecting a few rows because successful cleaning requires validation of the complete datasets rather than only sample observations.

## Principle

A cleaning transformation should satisfy two conditions:

1. the intended problem is corrected;
2. unrelated information is preserved.

Therefore, cleaning validation is based on the idea of **controlled change**.

For example:

- `StateHoliday` should change in representation but not in semantic category counts;
- `Date` should change in data type but not in calendar meaning;
- missing competition information should remain missing;
- untouched variables such as `Sales`, `Customers`, and `Promo` should remain exactly the same.

Validation before saving prevents accidental modifications from becoming part of the processed dataset.

## Evaluation Criteria

The cleaned datasets are ready to be saved if:

1. row and column counts are preserved;
2. no duplicate records or duplicate business keys are introduced;
3. `StateHoliday` contains only `0`, `a`, `b`, and `c` using one consistent string type;
4. `Date` is stored as a datetime type;
5. `Date` remains fully consistent with `DayOfWeek`;
6. the expected missing-value patterns are preserved;
7. the two datasets still contain identical store-ID sets;
8. fields that were not intended to change remain identical to the raw versions.

All validation checks should return either `True`, `0`, or the expected known count before the cleaned datasets are written to `data/processed/`.

In [7]:
# Perform final validation before saving the cleaned datasets.

# ------------------------------------------------------------
# 1. Check structural preservation
# ------------------------------------------------------------

train_rows_preserved = (
    train_clean.shape[0] == train.shape[0]
)

train_columns_preserved = (
    train_clean.shape[1] == train.shape[1]
)

store_rows_preserved = (
    store_clean.shape[0] == store.shape[0]
)

store_columns_preserved = (
    store_clean.shape[1] == store.shape[1]
)


# ------------------------------------------------------------
# 2. Check duplicates and business keys
# ------------------------------------------------------------

train_full_duplicates = int(
    train_clean.duplicated().sum()
)

store_full_duplicates = int(
    store_clean.duplicated().sum()
)

train_key_duplicates = int(
    train_clean.duplicated(
        subset=["Store", "Date"]
    ).sum()
)

store_key_duplicates = int(
    store_clean.duplicated(
        subset=["Store"]
    ).sum()
)


# ------------------------------------------------------------
# 3. Validate StateHoliday
# ------------------------------------------------------------

stateholiday_allowed = {
    "0",
    "a",
    "b",
    "c"
}

stateholiday_invalid = int(
    (
        train_clean["StateHoliday"].notna()
        & ~train_clean["StateHoliday"].isin(
            stateholiday_allowed
        )
    ).sum()
)

stateholiday_dtype = str(
    train_clean["StateHoliday"].dtype
)


# ------------------------------------------------------------
# 4. Validate Date
# ------------------------------------------------------------

date_is_datetime = (
    pd.api.types.is_datetime64_any_dtype(
        train_clean["Date"]
    )
)

expected_day_of_week = (
    train_clean["Date"].dt.dayofweek + 1
)

weekday_mismatches = int(
    (
        train_clean["DayOfWeek"]
        != expected_day_of_week
    ).sum()
)


# ------------------------------------------------------------
# 5. Validate missing-value structure
# ------------------------------------------------------------

train_missing_total = int(
    train_clean.isna().sum().sum()
)

promo2_missing_correct = (
    int(store_clean["Promo2SinceWeek"].isna().sum()) == 544
    and int(store_clean["Promo2SinceYear"].isna().sum()) == 544
    and int(store_clean["PromoInterval"].isna().sum()) == 544
)

competition_missing_correct = (
    int(store_clean["CompetitionDistance"].isna().sum()) == 3
    and int(store_clean["CompetitionOpenSinceMonth"].isna().sum()) == 354
    and int(store_clean["CompetitionOpenSinceYear"].isna().sum()) == 354
)


# ------------------------------------------------------------
# 6. Validate Store ID consistency
# ------------------------------------------------------------

clean_train_stores = set(
    train_clean["Store"].unique()
)

clean_store_stores = set(
    store_clean["Store"].unique()
)

store_ids_still_identical = (
    clean_train_stores == clean_store_stores
)


# ------------------------------------------------------------
# 7. Verify that untouched variables were preserved
# ------------------------------------------------------------

train_untouched_cols = [
    "Store",
    "DayOfWeek",
    "Sales",
    "Customers",
    "Open",
    "Promo",
    "SchoolHoliday"
]

store_untouched_cols = [
    "Store",
    "StoreType",
    "Assortment",
    "Promo2"
]

train_untouched_preserved = (
    train[train_untouched_cols]
    .equals(
        train_clean[train_untouched_cols]
    )
)

store_untouched_preserved = (
    store[store_untouched_cols]
    .equals(
        store_clean[store_untouched_cols]
    )
)


# ------------------------------------------------------------
# 8. Build final validation table
# ------------------------------------------------------------

final_cleaning_validation = pd.DataFrame({
    "Check": [
        "Train rows preserved",
        "Train columns preserved",
        "Store rows preserved",
        "Store columns preserved",
        "Train full-row duplicates",
        "Store full-row duplicates",
        "Train Store-Date duplicates",
        "Store ID duplicates",
        "StateHoliday dtype",
        "StateHoliday invalid categories",
        "Date is datetime",
        "DayOfWeek mismatches",
        "Train total missing values",
        "Promo2 missing pattern preserved",
        "Competition missing pattern preserved",
        "Store ID sets remain identical",
        "Untouched train fields preserved",
        "Untouched store fields preserved"
    ],
    "Result": [
        train_rows_preserved,
        train_columns_preserved,
        store_rows_preserved,
        store_columns_preserved,
        train_full_duplicates,
        store_full_duplicates,
        train_key_duplicates,
        store_key_duplicates,
        stateholiday_dtype,
        stateholiday_invalid,
        date_is_datetime,
        weekday_mismatches,
        train_missing_total,
        promo2_missing_correct,
        competition_missing_correct,
        store_ids_still_identical,
        train_untouched_preserved,
        store_untouched_preserved
    ]
})

display(final_cleaning_validation)


# ------------------------------------------------------------
# 9. Display final cleaned dtypes
# ------------------------------------------------------------

train_clean_dtypes = pd.DataFrame({
    "Column": train_clean.columns,
    "Dtype": train_clean.dtypes.astype(str).values
})

store_clean_dtypes = pd.DataFrame({
    "Column": store_clean.columns,
    "Dtype": store_clean.dtypes.astype(str).values
})

print("\nFinal train_clean data types:")
display(train_clean_dtypes)

print("\nFinal store_clean data types:")
display(store_clean_dtypes)

,Check,Result
0,Train rows preserved,True
1,Train columns preserved,True
2,Store rows preserved,True
3,Store columns preserved,True
4,Train full-row duplicates,0
5,Store full-row duplicates,0
6,Train Store-Date duplicates,0
7,Store ID duplicates,0
8,StateHoliday dtype,string
9,StateHoliday invalid categories,0



Final train_clean data types:


,Column,Dtype
0,Store,int64
1,DayOfWeek,int64
2,Date,datetime64[us]
3,Sales,int64
4,Customers,int64
5,Open,int64
6,Promo,int64
7,StateHoliday,string
8,SchoolHoliday,int64



Final store_clean data types:


,Column,Dtype
0,Store,int64
1,StoreType,str
2,Assortment,str
3,CompetitionDistance,Float64
4,CompetitionOpenSinceMonth,Int64
5,CompetitionOpenSinceYear,Int64
6,Promo2,int64
7,Promo2SinceWeek,Int64
8,Promo2SinceYear,Int64
9,PromoInterval,string


# 7. Save and Verify the Cleaned Datasets

## Purpose

After all intended cleaning transformations and post-cleaning validation checks have passed, this section writes the cleaned datasets to the processed-data directory.

The original files in `data/raw/` are not overwritten.

Two new files are created:

- `data/processed/train_clean.csv`
- `data/processed/store_clean.csv`

After saving, both files are read back from disk and validated again. This final step confirms that the exported files can be successfully reused by later notebooks.

## Input

The files are generated from the cleaned in-memory DataFrames:

- `train_clean`
- `store_clean`

These DataFrames were created from the original raw datasets and have passed the final cleaning validation.

## Output

The following new files are created:

- `data/processed/train_clean.csv`
- `data/processed/store_clean.csv`

The original files remain unchanged:

- `data/raw/train.csv`
- `data/raw/store.csv`

The exported CSV files will be used as the input data for subsequent exploratory analysis and feature-engineering stages.

## Method

`DataFrame.to_csv()` is used with `index=False` so that the Pandas row index is not written as an additional data column.

The cleaned `Date` values are written using the standard `YYYY-MM-DD` representation.

After export, `pd.read_csv()` is used to reload both processed files.

Because CSV is a text-based format and does not preserve Pandas data types, important data types are explicitly restored during reloading:

- `StateHoliday` is restored as a Pandas string;
- `Date` is restored as datetime;
- nullable competition and `Promo2` variables are restored using nullable numeric types;
- `PromoInterval` is restored as a Pandas string.

The reloaded files are then compared with the in-memory cleaned datasets in terms of dimensions, key fields, missing-value counts, category values, and date range.

## Principle

Saving a DataFrame successfully does not guarantee that the resulting file can be reconstructed correctly.

CSV stores values as text and does not preserve metadata such as:

- Pandas nullable integer types;
- Pandas string types;
- datetime types.

Therefore, a reproducible workflow should verify not only the in-memory cleaned dataset but also the exported file that will actually be used later.

The process is:

**Clean → Validate → Save → Reload → Validate again**

This ensures that the processed files represent the intended cleaned data and can serve as reliable inputs for subsequent analysis.

## Evaluation Criteria

The export is considered successful if:

1. both processed files are created;
2. neither raw file is modified;
3. the reloaded datasets have the same dimensions as the cleaned in-memory datasets;
4. store identifiers remain unchanged;
5. `StateHoliday` still contains only `0`, `a`, `b`, and `c`;
6. the date range remains unchanged;
7. missing-value counts remain unchanged;
8. business-key uniqueness remains intact.

If all checks pass, the processed datasets are ready for use in later notebooks.

In [8]:
# Save the cleaned datasets and verify the exported files.

# ------------------------------------------------------------
# 1. Define processed-data output paths
# ------------------------------------------------------------

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TRAIN_CLEAN_PATH = (
    PROCESSED_DIR / "train_clean.csv"
)

STORE_CLEAN_PATH = (
    PROCESSED_DIR / "store_clean.csv"
)


# ------------------------------------------------------------
# 2. Save the cleaned datasets
# ------------------------------------------------------------

train_clean.to_csv(
    TRAIN_CLEAN_PATH,
    index=False,
    date_format="%Y-%m-%d"
)

store_clean.to_csv(
    STORE_CLEAN_PATH,
    index=False
)


print("Cleaned datasets saved successfully.")
print(f"train_clean.csv: {TRAIN_CLEAN_PATH}")
print(f"store_clean.csv: {STORE_CLEAN_PATH}")


# ------------------------------------------------------------
# 3. Reload train_clean.csv
# ------------------------------------------------------------

train_saved = pd.read_csv(
    TRAIN_CLEAN_PATH,
    dtype={
        "StateHoliday": "string"
    }
)

train_saved["Date"] = pd.to_datetime(
    train_saved["Date"],
    format="%Y-%m-%d",
    errors="raise"
)


# ------------------------------------------------------------
# 4. Reload store_clean.csv with intended data types
# ------------------------------------------------------------

store_saved = pd.read_csv(
    STORE_CLEAN_PATH,
    dtype={
        "CompetitionDistance": "Float64",
        "CompetitionOpenSinceMonth": "Int64",
        "CompetitionOpenSinceYear": "Int64",
        "Promo2SinceWeek": "Int64",
        "Promo2SinceYear": "Int64",
        "PromoInterval": "string"
    }
)


# ------------------------------------------------------------
# 5. Validate the exported train dataset
# ------------------------------------------------------------

train_saved_store_ids = set(
    train_saved["Store"].unique()
)

train_clean_store_ids = set(
    train_clean["Store"].unique()
)

train_export_validation = {
    "Shape preserved":
        train_saved.shape == train_clean.shape,

    "Store IDs preserved":
        train_saved_store_ids
        == train_clean_store_ids,

    "Store-Date duplicates":
        int(
            train_saved.duplicated(
                subset=["Store", "Date"]
            ).sum()
        ) == 0,

    "Missing count preserved":
        int(train_saved.isna().sum().sum())
        == int(train_clean.isna().sum().sum()),

    "StateHoliday categories preserved":
        set(
            train_saved["StateHoliday"]
            .dropna()
            .unique()
        )
        ==
        set(
            train_clean["StateHoliday"]
            .dropna()
            .unique()
        ),

    "Earliest date preserved":
        train_saved["Date"].min()
        == train_clean["Date"].min(),

    "Latest date preserved":
        train_saved["Date"].max()
        == train_clean["Date"].max()
}


# ------------------------------------------------------------
# 6. Validate the exported store dataset
# ------------------------------------------------------------

store_saved_ids = set(
    store_saved["Store"].unique()
)

store_clean_ids = set(
    store_clean["Store"].unique()
)

store_export_validation = {
    "Shape preserved":
        store_saved.shape == store_clean.shape,

    "Store IDs preserved":
        store_saved_ids
        == store_clean_ids,

    "Store ID duplicates":
        int(
            store_saved.duplicated(
                subset=["Store"]
            ).sum()
        ) == 0,

    "Missing counts preserved":
        store_saved.isna().sum().equals(
            store_clean.isna().sum()
        ),

    "Promo2SinceWeek dtype":
        str(
            store_saved["Promo2SinceWeek"].dtype
        ) == "Int64",

    "Promo2SinceYear dtype":
        str(
            store_saved["Promo2SinceYear"].dtype
        ) == "Int64",

    "CompetitionDistance dtype":
        str(
            store_saved["CompetitionDistance"].dtype
        ) == "Float64",

    "CompetitionOpenSinceMonth dtype":
        str(
            store_saved[
                "CompetitionOpenSinceMonth"
            ].dtype
        ) == "Int64",

    "CompetitionOpenSinceYear dtype":
        str(
            store_saved[
                "CompetitionOpenSinceYear"
            ].dtype
        ) == "Int64"
}


# ------------------------------------------------------------
# 7. Build export-validation summary
# ------------------------------------------------------------

export_validation = pd.DataFrame(
    [
        {
            "Dataset": "train_clean",
            "Check": check,
            "Result": result
        }
        for check, result
        in train_export_validation.items()
    ]
    +
    [
        {
            "Dataset": "store_clean",
            "Check": check,
            "Result": result
        }
        for check, result
        in store_export_validation.items()
    ]
)

print("\nExport validation:")
display(export_validation)


# ------------------------------------------------------------
# 8. Confirm output files
# ------------------------------------------------------------

output_files = pd.DataFrame({
    "File": [
        "train_clean.csv",
        "store_clean.csv"
    ],
    "Exists": [
        TRAIN_CLEAN_PATH.exists(),
        STORE_CLEAN_PATH.exists()
    ],
    "Size (MB)": [
        round(
            TRAIN_CLEAN_PATH.stat().st_size
            / (1024 ** 2),
            2
        ),
        round(
            STORE_CLEAN_PATH.stat().st_size
            / (1024 ** 2),
            2
        )
    ]
})

print("\nProcessed output files:")
display(output_files)

Cleaned datasets saved successfully.
train_clean.csv: d:\Coding\NTUHomeworks\AssignmentOfCA6000\data\processed\train_clean.csv
store_clean.csv: d:\Coding\NTUHomeworks\AssignmentOfCA6000\data\processed\store_clean.csv

Export validation:


,Dataset,Check,Result
0,train_clean,Shape preserved,True
1,train_clean,Store IDs preserved,True
2,train_clean,Store-Date duplicates,True
3,train_clean,Missing count preserved,True
4,train_clean,StateHoliday categories preserved,True
5,train_clean,Earliest date preserved,True
6,train_clean,Latest date preserved,True
7,store_clean,Shape preserved,True
8,store_clean,Store IDs preserved,True
9,store_clean,Store ID duplicates,True



Processed output files:


,File,Exists,Size (MB)
0,train_clean.csv,True,33.38
1,store_clean.csv,True,0.04


# 8. Summary of Data Cleaning

The Rossmann Store Sales datasets were cleaned using an evidence-based approach based on the findings established in `01_check.ipynb`.

The cleaning process focused on correcting confirmed representation issues while preserving valid observations, structural missingness, and genuinely unavailable information.

## `train` Dataset

The original `train.csv` contained 1,017,209 daily observations and 9 variables.

No rows or columns were removed during cleaning.

### `StateHoliday`

The main confirmed representation issue was found in `StateHoliday`.

The no-holiday category was represented using both:

- integer `0`;
- string `"0"`.

These two representations had the same business meaning but were treated as different Python values.

The entire column was therefore normalized to Pandas' `string` data type.

After normalization, the variable contains four consistent categories:

- `0`
- `a`
- `b`
- `c`

The category frequencies were preserved, and no new missing or invalid values were introduced.

### `Date`

The original `Date` column was stored as a string.

Because the inspection stage confirmed that all date values were valid and followed the expected `YYYY-MM-DD` format, the column was converted to Pandas datetime format.

The conversion preserved:

- all 1,017,209 rows;
- the original date range from `2013-01-01` to `2015-07-31`;
- complete consistency with `DayOfWeek`.

No invalid or missing dates were introduced.

### Retained Observations

No sales or customer records were removed.

The rare open-store zero-sales observations identified during inspection were retained because they were unusual but not confirmed as data errors.

Likewise, IQR-based potential outliers in `Sales` and `Customers` were retained because statistical extremity alone was not considered sufficient evidence for removal.

## `store` Dataset

The original `store.csv` contained 1,115 store records and 10 variables.

No rows or columns were removed during cleaning.

### `Promo2` Variables

The missing values in:

- `Promo2SinceWeek`
- `Promo2SinceYear`
- `PromoInterval`

were preserved.

All 544 stores with `Promo2 = 0` had all three detail fields missing, while all 571 stores with `Promo2 = 1` had complete detail information.

This pattern was identified as structural missingness rather than accidental data loss.

The variables were normalized to more appropriate data types:

- `Promo2SinceWeek` → nullable `Int64`
- `Promo2SinceYear` → nullable `Int64`
- `PromoInterval` → Pandas `string`

No statistical imputation was applied.

### Competition Variables

The competition-related missing values were also preserved because they represent genuinely unavailable information.

The cleaned dataset retains:

- 3 missing `CompetitionDistance` values;
- 354 missing `CompetitionOpenSinceMonth` values;
- 354 missing `CompetitionOpenSinceYear` values.

The opening month and year remain missing together in all 354 affected stores.

Among these stores, 351 still contain a valid competition distance, while 3 stores contain no competition information in any of the three fields.

The variables were normalized as:

- `CompetitionDistance` → nullable `Float64`
- `CompetitionOpenSinceMonth` → nullable `Int64`
- `CompetitionOpenSinceYear` → nullable `Int64`

No artificial competition distance or opening date was introduced.

## Post-Cleaning Validation

A complete post-cleaning validation was performed before export.

The validation confirmed that:

- all original rows and columns were preserved;
- no full-row duplicates were introduced;
- no duplicate `Store-Date` business keys were introduced in `train_clean`;
- no duplicate `Store` identifiers were introduced in `store_clean`;
- `StateHoliday` contains only valid normalized categories;
- `Date` is stored as datetime and remains consistent with `DayOfWeek`;
- the original missing-value patterns were preserved;
- both datasets still contain identical sets of 1,115 store IDs;
- all variables that were not intentionally transformed remained unchanged.

## Processed Output

The cleaned datasets were saved as new files:

- `data/processed/train_clean.csv`
- `data/processed/store_clean.csv`

The original files in `data/raw/` were not modified.

Both processed files were subsequently reloaded from disk and validated again.

The reloaded datasets preserved:

- dataset dimensions;
- store identifiers;
- business-key uniqueness;
- missing-value counts;
- `StateHoliday` categories;
- date boundaries;
- the intended nullable numeric data types.

The processed files are therefore considered ready for subsequent exploratory data analysis and feature engineering.

## Cleaning Outcome

The cleaning stage required relatively limited correction because the raw Rossmann datasets showed generally high data quality.

The primary cleaning actions were:

1. normalizing the mixed representation of `StateHoliday`;
2. converting `Date` to an appropriate datetime type;
3. normalizing nullable numeric and string data types in `store`;
4. preserving structural and genuinely unknown missing values without inappropriate imputation.

No observations were removed solely because they were statistically unusual.

The resulting processed datasets preserve the original information while providing more consistent and appropriate data representations for later analysis.